# 18 · Hybrid search, reranking and agentic RAG with citations
Level L3 · Part 3 · core · ~30 min · builds on 17

## Goal
Search the FTC mail-order guide two ways (keyword and vector), fuse the two lists, rerank the best candidates, and let
an agent search as a tool, capped at 2 searches. Its answer must cite the passages it read, and the notebook checks
each citation in code.

## Why it matters
The support desk gets questions in two styles. Some reuse the guide's exact words ("seven working days"). Others use
the customer's words ("get my money back", where the guide says "refund"). Keyword search alone misses the second
style. **Nothing is sent to a model here.** Hybrid search returns both styles in one ranked list. You do not need it
when the whole guide fits in the prompt (then no retrieval is needed), or when every question uses the guide's words
(then one index is enough).

In [1]:
from pathlib import Path
import re
import sqlite3
from langchain_text_splitters import RecursiveCharacterTextSplitter  # the same splitter and chunks as lesson 17

doc = Path("../data/ftc_30_day_rule.txt").read_text(encoding="utf-8")  # the FTC guide (see data/SOURCES.md)
splitter = RecursiveCharacterTextSplitter(separators=["\n\n"], chunk_size=120, chunk_overlap=0, keep_separator=False,
                                          length_function=lambda text: len(text.split()))
chunks = splitter.split_text(doc)  # whole paragraphs, up to 120 words each

db = sqlite3.connect(":memory:")  # SQLite (standard library) with FTS5, its full-text index
db.execute("CREATE VIRTUAL TABLE chunks USING fts5(text)")
db.executemany("INSERT INTO chunks(rowid, text) VALUES (?, ?)", list(enumerate(chunks)))


def keyword_hits(query: str, k: int = 8) -> list[tuple[int, float]]:
    """BM25 search: each word is quoted (FTS5 reads dashes and quotes as syntax); bm25() is lower for a better match."""
    match = " OR ".join(f'"{word}"' for word in re.findall(r"\w+", query))
    return db.execute("SELECT rowid, bm25(chunks) FROM chunks WHERE chunks MATCH ? ORDER BY 2 LIMIT ?", (match, k)).fetchall()


for q in ["seven working days", "get my money back"]:
    top = keyword_hits(q, k=1)[0][0]
    print(f"keyword · {q!r} → top chunk {top}:\n{chunks[top]}\n")

keyword · 'seven working days' → top chunk 26:
- If the customer paid by cash, check, money order, or by credit where a third party is the creditor, or by any other method except credit where you are a creditor, you must refund the correct amount within seven working days after the order is cancelled.

- If the customer paid by credit where you are a creditor, you must credit the customter's account or notify the customer that the account will not be charged within one billing cycle after the order is cancelled.

How Much You Must Refund

keyword · 'get my money back' → top chunk 53:
Qualifying 30-day or Other Shipment Representations

Q: In soliciting telephone orders we make no shipment representation, so the 30-day rule applies. In taking the order, the sales representative tells the customer that the merchandise will be shipped in 72 hours. Then we discover that the merchandise cannot be shipped in 72 hours, but can be shipped within 30 days. Do we have to get the customer’s agreem

The first query finds the refund rule because the guide uses the same words (its paragraph contains "seven working
days"). The second query fails: the guide never says "money back", so the keyword search returns passages that share
a common word, not the refund rules. Keyword search matches words, not meaning. Vector search does the reverse; the
next steps combine them.

## The idea
**Keyword search (BM25)** scores a chunk by the query words it contains, weighting rare words more. **Vector search**
(lesson 17) scores by meaning. **Hybrid search** runs both and merges the two ranked lists with **reciprocal rank
fusion**: a chunk gets 1 divided by (60 plus its rank) from each list, and the sums are sorted. Ranks are used, not
raw scores, because BM25 and cosine scores are on different scales.

A **reranker** reads each (question, passage) pair more closely and reorders a short list, here 8 candidates down to 3.
An **agent** with search as a tool writes its own query, then searches or answers. Its answer cites passage
numbers, and the code checks that each one was really shown.

Analogy: a librarian with two catalogues, a title index (keyword) and a subject index (meaning). She merges two short
lists, a colleague picks the best three, and she checks the shelves again if the answer is missing.

## Picture
```mermaid
flowchart LR
  Q[Customer question] --> A[Agent writes a search query]
  A --> K[Keyword: BM25 over FTS5]
  A --> V[Vector: cosine in Chroma]
  K --> F[Fuse the two ranks: RRF, 8 candidates]
  V --> F
  F --> R[Reranker: model keeps the best 3]
  R --> A
  A -- passages enough? --> C[Answer with citations]
  C --> CK[Check: every cited number was shown]
```

## Step by step
**Step 1 · the vector store (no model call).** The chunk vectors were computed once, when lesson 17 was recorded, and
saved in `data/ftc_chunks_embeddings.json`. This cell checks that they match the chunks made above, then loads them
into Chroma, which ranks by cosine similarity. Nothing is embedded here.

In [2]:
import json
import chromadb
from chromadb.config import Settings

saved = json.loads(Path("../data/ftc_chunks_embeddings.json").read_text(encoding="utf-8"))
assert [c["text"] for c in saved["chunks"]] == chunks, "the saved vectors were made from different chunks"
store = chromadb.EphemeralClient(settings=Settings(anonymized_telemetry=False)).create_collection(
    "ftc_guide", embedding_function=None, configuration={"hnsw": {"space": "cosine"}})  # our own vectors: no model call
store.add(ids=[str(i) for i in range(len(chunks))], documents=chunks, embeddings=[c["vector"] for c in saved["chunks"]])
print("model:", saved["model"], "· indexed chunks:", store.count())

model: gemini-embedding-2 · indexed chunks: 56


**Step 2 · the clients and the call budget.** `gemini` sends generation to Vertex AI. `embedder` calls the Gemini API
for embeddings, as in lesson 17. Both read the same key from `.env`, never typed here. While a recording replays, a
placeholder is used and no key is sent. The budget counts every model request, generation and embedding. The recorded
run: 5 calls (3 generation + 2 embedding). Live, a second search could add more (up to 8 in total). The code stops at
two searches, so the plan cannot grow without limit.

In [3]:
# No key? Replay mode runs this notebook as recorded. Live mode needs GOOGLE_CLOUD_API_KEY in the topic's .env file.
import sys; sys.path.insert(0, "..")
import llm_replay  # course helper: records and replays model calls (not part of the SDK)
llm_replay.start("18_hybrid-agentic-rag")

from functools import lru_cache, partial
from google import genai  # the official Google SDK
from google.genai import errors, types
from tenacity import retry, retry_if_exception, stop_after_attempt, wait_exponential_jitter
from tut_env import env  # course helper: reads a setting from the .env file (not part of the SDK)

KEY = "replay" if llm_replay.is_replaying() else env("GOOGLE_CLOUD_API_KEY")  # a placeholder while replaying
gemini = genai.Client(vertexai=True, api_key=KEY, http_options=types.HttpOptions(timeout=30_000))  # generation
embedder = genai.Client(api_key=KEY, http_options=types.HttpOptions(timeout=30_000))  # embeddings, as in 17
GEN_MODEL, EMBED_MODEL, DIMS = "gemini-3.1-flash-lite", "gemini-embedding-2", 768
QUERY_PREFIX = "task: search result | query: "  # questions get the query prefix; the chunks got the document prefix

LLM mode: replay (18_hybrid-agentic-rag.yaml)


**Retries and the vector search.** A 429 or 5xx is retried with growing waits (`call`). `vector_hits` embeds a question
and asks Chroma for the 8 nearest chunks. It is cached, so a question that two cells ask about is embedded once.

In [4]:
@retry(retry=retry_if_exception(lambda e: isinstance(e, errors.APIError) and e.code in {408, 429, 500, 502, 503, 504}),
       stop=stop_after_attempt(4), wait=wait_exponential_jitter(initial=5, max=30, jitter=5), reraise=True)
def call(make):
    """Runs one request and retries 429 and 5xx with growing waits, as in lesson 02."""
    return make()


@lru_cache(maxsize=None)  # a question is embedded once, even when two cells ask for it
def vector_hits(query: str) -> tuple[tuple[int, float], ...]:
    vec = call(lambda: embedder.models.embed_content(
        model=EMBED_MODEL, contents=[types.Content(parts=[types.Part.from_text(text=QUERY_PREFIX + query)])],
        config=types.EmbedContentConfig(output_dimensionality=DIMS))).embeddings[0].values
    res = store.query(query_embeddings=[vec], n_results=8)
    return tuple((int(i), 1 - d) for i, d in zip(res["ids"][0], res["distances"][0]))  # similarity = 1 - distance

**Step 3 · fuse the two lists (one embedding request, no generation).** `fuse` is reciprocal rank fusion. It is a few
lines because the two lists come from two engines (SQLite and Chroma), and no one call merges them. A chunk found by
both lists rises to the top. Run the three lists for the customer's question and read which chunks only one list found.

In [5]:
Q1 = ("A customer ordered a jacket 20 days ago and the shop now says it will ship in 45 days. "
      "Can they cancel for a full refund?")


def fuse(lists: list[list[int]], k: int = 60) -> list[int]:
    """Reciprocal rank fusion: each list adds 1 / (k + rank) to a chunk (k = 60, from the original paper)."""
    score: dict[int, float] = {}
    for ranked in lists:
        for rank, cid in enumerate(ranked, start=1):
            score[cid] = score.get(cid, 0) + 1 / (k + rank)
    return sorted(score, key=score.get, reverse=True)


kw = [cid for cid, _ in keyword_hits(Q1)]  # keyword list: no model call
print("keyword :", kw)

keyword : [4, 25, 13, 9, 15, 16, 21, 49]


The keyword list needed no model. The vector list needs one embedding request for the question, which counts toward
the 5-call budget like any other request. Fuse the two lists and compare them.

In [6]:
vec = [cid for cid, _ in vector_hits(Q1)]  # vector list: one embedding request for the question
fused = fuse([kw, vec])[:8]
print("vector  :", vec)
print("fused   :", fused)
print("only the keyword list:", sorted(set(kw) - set(vec)), "· only the vector list:", sorted(set(vec) - set(kw)))

vector  : [21, 8, 15, 47, 9, 27, 12, 13]
fused   : [21, 15, 9, 13, 4, 25, 8, 47]
only the keyword list: [4, 16, 25, 49] · only the vector list: [8, 12, 27, 47]


Read the three lists. The chunks found by only one list are the reason to keep both. Keyword search catches the exact
words in the question. Vector search catches the passages that talk about the same thing in other words. The fused list
keeps chunks that both lists found near the top.

**Step 4 · the reranker (a generation request, used inside the search).** Fusion orders by rank, not by a close reading
of each passage. A reranker reads the question and each candidate together, and returns the candidates best first. It
is a model call with a schema: the reply must be a list of chunk numbers, which the code checks. Production systems
often use a dedicated reranker model for this step; this course uses its own model.

In [7]:
from pydantic import BaseModel  # the schema for the reranker's answer (no call)


class Ranking(BaseModel):
    best_first: list[int]  # the chunk numbers, most useful for answering the question first


RERANK_SYSTEM = "Rank the passages for a support question: the most direct answers first. Use only the numbers given."


def rerank(question: str, cids: list[int], keep: int = 3) -> list[int]:
    excerpts = "\n\n".join(f"[{c}] {chunks[c]}" for c in cids)
    resp = call(lambda: gemini.models.generate_content(
        model=GEN_MODEL, contents=f"Question: {question}\n\nPassages:\n{excerpts}",
        config=types.GenerateContentConfig(system_instruction=RERANK_SYSTEM, temperature=0,
                                           response_mime_type="application/json", response_schema=Ranking)))
    order = [c for c in Ranking.model_validate_json(resp.text).best_first if c in cids]  # ignore numbers not offered
    return order[:keep]

**Step 5 · the agent, with search as a tool.** The model sees one tool, `search_guide`. It writes the query, then decides
whether the passages are enough. The tool runs the hybrid search and the reranker, and returns the best 3 passages with
their numbers. The cap is 2 searches; past it the tool is removed, so the model must answer. Every search is printed.
In the recorded run the agent searched once and was satisfied, so the second-search path is not in the recording.

**The search tool.** The model reads the tool's name, description and argument schema, as in lesson 07. The
description says when to search again.

In [8]:
SEARCH = types.FunctionDeclaration(
    name="search_guide", description="Search the FTC mail-order rule guide. Returns the 3 best passages with their "
    "numbers. If they do not answer the question, search again with other words.",
    parameters_json_schema={"type": "object", "required": ["query"], "properties": {"query": {"type": "string",
        "description": "a short query in the guide's words, such as 'delay notice refund'"}}})
SYSTEM = ("You are the Brightside Outfitters support desk. Answer only from passages returned by search_guide. "
          "Cite the passage numbers you used, like [8]. If the passages do not answer, say so and do not guess.")
MAX_SEARCHES = 2  # the cap: each search is one embedding and one rerank
seen: set[int] = set()  # every passage the agent was shown, for the citation check


def search_guide(question: str, query: str) -> list[dict]:
    cids = fuse([[c for c, _ in keyword_hits(query)], [c for c, _ in vector_hits(query)]])[:8]  # hybrid: keyword and vector, fused, 8 candidates
    top = rerank(question, cids)  # the reranker judges against the customer's question, not the search words
    seen.update(top)
    print(f"search {query!r}\n   fused {cids}\n   reranked to {top}")
    return [{"passage": c, "text": chunks[c]} for c in top]

**The loop.** The model answers, or asks for a search. Each search goes through `search_guide`, and its result goes back
to the model as a function response. Past the cap, the search tool is removed, so the model must answer.

In [9]:
# The recorded run searches once and answers. The second-search path is not shown in the recording.
def answer(question: str) -> str:
    ask = partial(search_guide, question)  # the tool gets the question, so the reranker judges against it
    contents = [types.Content(role="user", parts=[types.Part.from_text(text=question)])]
    searches = 0
    while True:
        tools = [types.Tool(function_declarations=[SEARCH])] if searches < MAX_SEARCHES else None
        cfg = types.GenerateContentConfig(system_instruction=SYSTEM, temperature=0, tools=tools)
        resp = call(lambda: gemini.models.generate_content(model=GEN_MODEL, contents=contents, config=cfg))
        contents.append(resp.candidates[0].content)
        if not resp.function_calls:
            return resp.text  # the answer, with its citations
        parts = []
        for c in resp.function_calls:
            out = {"error": "search limit reached; answer now"}  # a request past the cap gets a result, not a search
            if searches < MAX_SEARCHES:
                searches += 1
                out = {"passages": ask(**dict(c.args))}
            parts.append(types.Part.from_function_response(name=c.name, response=out))
        contents.append(types.Content(role="user", parts=parts))

**Run the agent on the customer's question.** The recorded run makes 5 model calls: the agent's requests, the reranker
and the embeddings.

In [10]:
final = answer(Q1)  # the recorded run makes 5 model calls
cited = {int(n) for n in re.findall(r"\[(\d+)\]", final)}
print("\nanswer:", final)
print("cited:", sorted(cited), "· every cited passage was shown to the agent:", cited <= seen)
llm_replay.stop()  # the last model call of this notebook; nothing below calls a model

search 'cancel order delay refund'
   fused [9, 8, 21, 20, 25, 10, 15, 26]
   reranked to [10, 8, 21]

answer: Yes, you have the right to cancel the order for a full and prompt refund. When a seller cannot ship on time and proposes a delay, they must provide you with a notice that includes a statement informing you that you can cancel the order and obtain a full refund if you choose not to wait [10]. Furthermore, if you are notified of a revised shipment date of more than 30 days, you have the option to cancel the order [21].
cited: [10, 21] · every cited passage was shown to the agent: True


The agent's first search used its own words ("cancel order delay refund"), not the customer's question, so its fused
list differs from the list in the cell above. Fusion put chunk 9 first. The reranker dropped it and moved chunk 10 to
the top: the passage that states the full refund right.

**Step 6 · read the cited passages (no model call).** A citation proves that the number was shown. It does not prove
that the sentence matches the passage. Open each cited passage and compare it with the claim.

In [11]:
# Uses `cited` from the agent cell above, so run it after that cell.
for c in sorted(cited):
    print(f"[{c}] {chunks[c]}\n")

[10] In seeking your customer’s consent to delay, the first delay notice you provide to the customer (the "delay option" notice) must include:

- a definite revised shipment date or, if unknown, a statement that you are unable to provide a revised shipment date;

- a statement that, if the customer chooses not to wait, the customer can cancel the order and obtain a full and prompt refund; and

- some means for the customer to choose to cancel at your expense (e.g., by providing a postage prepaid reply card, toll-free telephone number, or website).

- the following information when you cannot provide a revised shipping date:
- the reason for the delay, and

[21] You must cancel an order and provide a prompt refund when:

- the customer exercises any option to cancel before you ship the merchandise;

- the customer does not respond to your first notice of a definite revised shipment date of 30 days or less and you have not shipped the merchandise or received the customer’s consent to a f

Compare the answer with the printed passages. Check the number of days, who pays and whether the customer can cancel.
A cited answer can still misstate a rule, so a person reads it before it reaches a customer. Lesson 22 turns this
reading into a labelled test set and a score.

## At work
- Anthropic's contextual retrieval post measured top-20 retrieval misses: 5.7% with plain embeddings, 2.9% after adding
  BM25 (with contextual chunks), 1.9% after adding a reranker. This lesson builds the BM25 and reranker steps, without
  the contextual chunks → lesson 22 measures the misses on your own questions.
- Teams check retrieval separately from the answer: did the right passage reach the top k? Only then do they judge the
  answer → lesson 08 (first eval) and lesson 22 (eval depth).
- Retrieved text comes from outside. A document can say "ignore your instructions", so treat it like user input
  → lesson 30 (injection and sandboxing).
- Go deeper: contextual chunks (a short context added to each chunk before it is embedded) and dedicated reranker
  models with their own latency budget.

## Common mistakes
1. **Passing the raw question to FTS5.** The apostrophe and the dash are FTS5 syntax, so a query with them fails. The
   error below is what the learner sees.

In [12]:
db.execute("SELECT rowid FROM chunks WHERE chunks MATCH ?", ("can't ship 30-day",)).fetchall()

OperationalError: fts5: syntax error near "'"

The fix is the one `keyword_hits` uses: take the words with `\w+` and quote each one.

2. **Adding BM25 scores to cosine scores.** The two numbers are on different scales, so one side wins whatever the
   question. The cell below prints one BM25 score and one cosine score for the same question. Fuse ranks, as `fuse` does.

In [13]:
print("BM25 (lower is better):", [(c, round(s, 2)) for c, s in keyword_hits(Q1)[:2]])
print("cosine (higher is better):", [(c, round(s, 2)) for c, s in vector_hits(Q1)[:2]])

BM25 (lower is better): [(4, -8.33), (25, -6.74)]
cosine (higher is better): [(21, 0.76), (8, 0.75)]


The BM25 numbers are negative and unbounded; the cosine numbers sit between -1 and 1. A score cannot show that an
answer is grounded. The cited passages can.

## Try it
Two ranked lists from two searches: `keyword = [5, 2, 9]` and `vector = [2, 7, 5]`. Compute the fused order by hand.
Each list adds 1 / (60 + rank) to a chunk, and ranks start at 1. Then check your order with `fuse`. No model call is needed.

In [14]:
keyword, vector = [5, 2, 9], [2, 7, 5]
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [15]:
print(fuse([keyword, vector]))  # [2, 5, 7, 9]: chunk 2 gets 1/62 + 1/61, chunk 5 gets 1/61 + 1/63

[2, 5, 7, 9]


## Recap
- Keyword search (BM25) finds the guide's exact words; vector search finds the same meaning in other words. Hybrid
  search keeps both, fused by rank.
- A reranker orders a short list of candidates by reading each one against the question.
- An agent with search as a tool can search again. Cap the searches (here 2), because each one costs model calls. In
  the recorded run it searched once and was satisfied.
- Check every citation in code, then read the passage yourself: a cited number proves only that it was shown.

**Interview:** "How would you improve retrieval when users use different words from the documents?" Strong answer:
"I would add keyword search next to vector search, fuse the two ranked lists, and rerank the top candidates. Then I would
measure recall at k (the share of labelled questions whose right passage is in the top k) before and after each change."